> **AI-assisted:** This notebook was created with AI assistance. Review paths and configuration before running in production.

# Extract MVP GeoJSON (single FED + DAs)

Colab notebook to stream-extract one federal electoral district (FED) and its dissemination areas (DAs) from `CRMP-full-data.zip` on the team Shared Drive.

**Output:** `single_fed_das.geojson` with `DGUID` and `C1_COUNT_TOTAL` (2021 population).

**After download:** place the file in `map-mvp/data/single_fed_das.geojson` in this repo.

In [ ]:
# ==========================================
# Cell 1: Environment initialization (Colab only)
# ==========================================
try:
    from google.colab import drive

    IN_COLAB = True
except ImportError:
    IN_COLAB = False

if not IN_COLAB:
    print("[WARN] Not running in Google Colab. Drive mount and browser download are skipped.")
    print("       Open this notebook in Colab (GitHub tab) and Runtime -> Run all.")
else:
    drive.mount("/content/drive")
    print("[OK] Google Drive mounted at /content/drive")

    get_ipython().run_line_magic("pip", "install -q geopandas fiona shapely pandas")
    print("[OK] geopandas, fiona, shapely, pandas installed")

In [ ]:
# ==========================================
# Cell 2: Stream-extract and merge data
# ==========================================
import os
import zipfile

import geopandas as gpd
import pandas as pd

# --- Configuration (adjust if needed) ---
SHARED_DRIVE_ROOT = "/content/drive/Shareddrives/CSCC01 - Group Project"
ZIP_FILENAME = "CRMP-full-data.zip"
ZIP_PATH = os.path.join(SHARED_DRIVE_ROOT, "Data", ZIP_FILENAME)

prov = "on"  # Ontario; see Raw_Data_Schema.md for other codes
fed_target = "35001"  # Ontario FEDUID; must match province (35xxx = ON)

# Paths inside the zip (from Raw_Data_Schema.md + Raw_Data_redist-mini-guide.md)
FED_REL = f"raw_data/statistics_canada/census_boundaries/{prov}/{prov}_electoral_districts_2023ro.gpkg"
DA_REL = f"raw_data/statistics_canada/census_boundaries/{prov}/{prov}_dissemination_areas.gpkg"
PROV_TO_DA_CSV = {
    "on": "raw_data/statistics_canada/census_profiles/profile_2021/006_dissemination_areas/ontario.csv",
    "qc": "raw_data/statistics_canada/census_profiles/profile_2021/006_dissemination_areas/quebec.csv",
    "bc": "raw_data/statistics_canada/census_profiles/profile_2021/006_dissemination_areas/bc.csv",
    "ab": "raw_data/statistics_canada/census_profiles/profile_2021/006_dissemination_areas/prairies.csv",
    "mb": "raw_data/statistics_canada/census_profiles/profile_2021/006_dissemination_areas/prairies.csv",
    "sk": "raw_data/statistics_canada/census_profiles/profile_2021/006_dissemination_areas/prairies.csv",
    "ns": "raw_data/statistics_canada/census_profiles/profile_2021/006_dissemination_areas/atlantic.csv",
    "nb": "raw_data/statistics_canada/census_profiles/profile_2021/006_dissemination_areas/atlantic.csv",
    "nl": "raw_data/statistics_canada/census_profiles/profile_2021/006_dissemination_areas/atlantic.csv",
    "pe": "raw_data/statistics_canada/census_profiles/profile_2021/006_dissemination_areas/atlantic.csv",
    "nt": "raw_data/statistics_canada/census_profiles/profile_2021/006_dissemination_areas/territories.csv",
    "nu": "raw_data/statistics_canada/census_profiles/profile_2021/006_dissemination_areas/territories.csv",
    "yt": "raw_data/statistics_canada/census_profiles/profile_2021/006_dissemination_areas/territories.csv",
}
CSV_REL = PROV_TO_DA_CSV.get(prov)
if CSV_REL is None:
    raise ValueError(f"[ERROR] Unknown province code '{prov}'. Update PROV_TO_DA_CSV.")

ZIP_VFS = f"/vsizip/{ZIP_PATH}/"
fed_gpkg = ZIP_VFS + FED_REL
da_gpkg = ZIP_VFS + DA_REL

print("=== Path configuration ===")
print(f"ZIP_PATH      : {ZIP_PATH}")
print(f"FED layer     : {FED_REL}")
print(f"DA layer      : {DA_REL}")
print(f"DA profile CSV: {CSV_REL}")
print(f"Target FEDUID : {fed_target} (province={prov})")

# --- Validate zip and inner paths ---
if not os.path.exists(ZIP_PATH):
    print(f"\n[ERROR] Zip file not found: {ZIP_PATH}")
    print("        Expected: Shared Drive -> CSCC01 - Group Project -> Data -> CRMP-full-data.zip")
    if os.path.isdir(SHARED_DRIVE_ROOT):
        data_dir = os.path.join(SHARED_DRIVE_ROOT, "Data")
        if os.path.isdir(data_dir):
            print(f"        Files in Data/: {os.listdir(data_dir)}")
        else:
            print(f"        Data/ folder missing under {SHARED_DRIVE_ROOT}")
    else:
        shareddrives = "/content/drive/Shareddrives"
        if os.path.isdir(shareddrives):
            print(f"        Available Shared Drives: {os.listdir(shareddrives)}")
    raise FileNotFoundError(ZIP_PATH)

print(f"\n[OK] Zip exists ({os.path.getsize(ZIP_PATH) / (1024**3):.2f} GB)")

with zipfile.ZipFile(ZIP_PATH, "r") as zf:
    names = set(zf.namelist())
    for label, rel in [("FED gpkg", FED_REL), ("DA gpkg", DA_REL), ("DA CSV", CSV_REL)]:
        if rel in names:
            print(f"[OK] {label} found in zip: {rel}")
        else:
            print(f"[ERROR] {label} NOT found at: {rel}")
            hint = rel.split("/")[-1]
            candidates = [n for n in names if hint in n]
            if candidates:
                print(f"        Similar paths ({len(candidates)}): {candidates[:5]}")
            else:
                print("        No similar paths. Manually confirm filename/path in the zip.")
            raise FileNotFoundError(rel)

# --- Read target FED boundary ---
print("\n=== Reading target FED boundary ===")
feds = gpd.read_file(fed_gpkg)
print(f"[OK] Loaded {len(feds)} FED polygons; CRS={feds.crs}")

if "FEDUID" not in feds.columns:
    print(f"[ERROR] Column FEDUID missing. Available: {list(feds.columns)}")
    raise KeyError("FEDUID")

feds["FEDUID"] = feds["FEDUID"].astype(str)
fed_match = feds[feds["FEDUID"] == str(fed_target)]
if fed_match.empty:
    sample = sorted(feds["FEDUID"].unique())[:10]
    print(f"[ERROR] FEDUID {fed_target} not found in {FED_REL}")
    print(f"        Sample FEDUID values: {sample} ...")
    raise ValueError(f"FEDUID {fed_target} not found")

target_fed_geom = fed_match.geometry.iloc[0]
fed_bbox = target_fed_geom.bounds
print(f"[OK] Target FED {fed_target} bbox (minx, miny, maxx, maxy): {fed_bbox}")

# --- Stream-prefilter DAs by bbox, then exact FED filter ---
print("\n=== Streaming DAs inside target FED ===")
das_in_bbox = gpd.read_file(da_gpkg, bbox=fed_bbox)
print(f"[OK] BBox prefilter returned {len(das_in_bbox)} DA polygons")

if "DGUID" not in das_in_bbox.columns:
    print(f"[ERROR] Column DGUID missing in DA layer. Available: {list(das_in_bbox.columns)}")
    raise KeyError("DGUID")

das_in_fed = das_in_bbox[das_in_bbox.geometry.centroid.within(target_fed_geom)].copy()
print(f"[OK] {len(das_in_fed)} DAs have centroids inside FED {fed_target}")
if len(das_in_fed) == 0:
    raise ValueError("No DAs matched the target FED. Check fed_target / province pairing.")

target_dguids = set(das_in_fed["DGUID"].astype(str))
print(f"[OK] Unique DGUIDs to join: {len(target_dguids)}")

# --- Chunk-read DA population CSV from inside zip ---
print("\n=== Chunk-reading 2021 DA population (CHARACTERISTIC_ID == 1) ===")
USECOLS = ["DGUID", "CHARACTERISTIC_ID", "C1_COUNT_TOTAL"]
pop_records = []
rows_scanned = 0
rows_kept = 0

with zipfile.ZipFile(ZIP_PATH, "r") as zf:
    with zf.open(CSV_REL) as raw_csv:
        try:
            chunk_iter = pd.read_csv(
                raw_csv,
                chunksize=20000,
                usecols=USECOLS,
                encoding="utf-8",
                low_memory=False,
            )
        except UnicodeDecodeError:
            print("[WARN] UTF-8 decode failed; retrying with latin-1")
            raw_csv.seek(0)
            chunk_iter = pd.read_csv(
                raw_csv,
                chunksize=20000,
                usecols=USECOLS,
                encoding="latin-1",
                low_memory=False,
            )

        for chunk in chunk_iter:
            rows_scanned += len(chunk)
            filtered = chunk[
                (chunk["CHARACTERISTIC_ID"] == 1)
                & (chunk["DGUID"].astype(str).isin(target_dguids))
            ]
            rows_kept += len(filtered)
            if not filtered.empty:
                pop_records.append(filtered)

print(f"[OK] Scanned {rows_scanned:,} CSV rows; kept {rows_kept:,} population rows")

if not pop_records:
    print("[ERROR] No population rows matched. Check CSV path and DGUID join key.")
    raise ValueError("Empty population extract")

pop_df = pd.concat(pop_records, ignore_index=True)
pop_df = pop_df.drop_duplicates(subset=["DGUID"], keep="first")
print(f"[OK] Population table: {len(pop_df)} unique DGUIDs")

# --- Merge geometry + population ---
print("\n=== Merging spatial geometry with census data ===")
das_in_fed["DGUID"] = das_in_fed["DGUID"].astype(str)
pop_df["DGUID"] = pop_df["DGUID"].astype(str)
final_gdf = das_in_fed.merge(pop_df[["DGUID", "C1_COUNT_TOTAL"]], on="DGUID", how="left")

missing_pop = final_gdf["C1_COUNT_TOTAL"].isna().sum()
if missing_pop:
    print(f"[WARN] {missing_pop} DAs have no population after join (left as null)")
else:
    print("[OK] All DAs have C1_COUNT_TOTAL")

print(f"[OK] Population range: {final_gdf['C1_COUNT_TOTAL'].min():.0f} – {final_gdf['C1_COUNT_TOTAL'].max():.0f}")

# Keep only required attributes for a smaller GeoJSON
final_gdf = final_gdf[["DGUID", "C1_COUNT_TOTAL", "geometry"]]
final_gdf = final_gdf.to_crs(epsg=4326)
print(f"[OK] Reprojected to EPSG:4326; {len(final_gdf)} features ready for export")

In [ ]:
# ==========================================
# Cell 3: Export GeoJSON and download
# ==========================================
import os
import sys

OUTPUT_NAME = "single_fed_das.geojson"
REPO_TARGET = "map-mvp/data/single_fed_das.geojson"

final_gdf.to_file(OUTPUT_NAME, driver="GeoJSON")
size_mb = os.path.getsize(OUTPUT_NAME) / (1024 * 1024)

print("=== Export summary ===")
print(f"File          : {OUTPUT_NAME}")
print(f"Features      : {len(final_gdf)}")
print(f"Columns       : {[c for c in final_gdf.columns if c != 'geometry']}")
print(f"File size     : {size_mb:.2f} MB")
print(f"Repo target   : {REPO_TARGET}")

if size_mb >= 5:
    print("[WARN] File is >= 5 MB. Definition of Done expects < 5 MB.")
else:
    print("[OK] File size under 5 MB")

required_cols = {"DGUID", "C1_COUNT_TOTAL"}
if required_cols.issubset(set(final_gdf.columns)):
    print(f"[OK] Required attributes present: {sorted(required_cols)}")
else:
    print(f"[ERROR] Missing columns. Expected {required_cols}, got {list(final_gdf.columns)}")

if "google.colab" in sys.modules:
    from google.colab import files

    files.download(OUTPUT_NAME)
    print("\n[OK] Browser download triggered.")
    print(f"     Move {OUTPUT_NAME} into your local repo at: {REPO_TARGET}")
else:
    print("\n[INFO] Not in Colab — file written locally only (no browser download).")